# Lab 02 · Exploratory Data Analysis & Preprocessing
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 2**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab02_EDA_and_Preprocessing.ipynb)

**In this lab you will:**
- load a real data set and inspect its structure
- visualise distributions, outliers and correlations
- handle missing values, encode categories and scale features
- reduce dimensions with PCA and t-SNE

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1 · Load the data
We use the **Titanic** passenger data (891 rows): survival, class, sex, age, fare … It contains missing values and mixed attribute types — ideal for practice.

In [ ]:
df = sns.load_dataset("titanic")
df.head()

In [ ]:
# ✏️ Optional: use your own data set instead of the built-in one
USE_MY_CSV = False   # change to True, run the cell, then choose a .csv file

if USE_MY_CSV:
    from google.colab import files          # works on Google Colab
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
    print("Loaded your file:", df.shape)
else:
    print("Using the built-in data set (Titanic).")

## 2 · First look: size, types, summary statistics

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
# Numeric summary (mean, std, quartiles) and categorical summary (counts, top value)
display(df.describe().round(2))
display(df.describe(include="object"))

## 3 · Missing values
Which columns have gaps, and how many?

In [ ]:
missing = df.isna().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
ax = missing.plot(kind="barh", color="tomato")
ax.set_xlabel("% missing"); ax.set_title("Missing values per column")
plt.show()

## 4 · Distributions and outliers

In [ ]:
num_cols = df.select_dtypes("number").columns.tolist()
print("Numeric columns:", num_cols)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[num_cols[min(2, len(num_cols)-1)]].dropna(), kde=True, ax=axes[0])
sns.boxplot(x=df[num_cols[-1]], ax=axes[1], color="lightblue")
axes[0].set_title("Histogram"); axes[1].set_title("Boxplot (dots = potential outliers)")
plt.show()

In [ ]:
# The IQR rule: values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR are potential outliers
col = num_cols[-1]                      # ✏️ choose another column
q1, q3 = df[col].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)]
print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f} -> {len(outliers)} potential outliers")

## 5 · Categorical attributes and relationships

In [ ]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Categorical columns:", cat_cols)
if "sex" in df and "survived" in df:
    sns.barplot(data=df, x="sex", y="survived", hue="class" if "class" in df else None)
    plt.title("Survival rate by sex and class"); plt.show()

In [ ]:
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation matrix (numeric attributes)"); plt.show()

## 6 · Cleaning: impute, encode, scale
We build the cleaned table step by step so that each operation is visible.

In [ ]:
clean = df.copy()

# 6.1 drop columns that are mostly empty or duplicate other columns
too_empty = [c for c in clean.columns if clean[c].isna().mean() > 0.5]
clean = clean.drop(columns=too_empty)
print("Dropped (more than 50% missing):", too_empty)

# 6.2 numeric -> median, categorical -> most frequent value
for c in clean.columns:
    if clean[c].dtype.kind in "biufc":
        clean[c] = clean[c].fillna(clean[c].median())
    else:
        clean[c] = clean[c].astype("object").fillna(clean[c].mode()[0])
print("Missing values left:", int(clean.isna().sum().sum()))

In [ ]:
# 6.3 one-hot encoding of categorical attributes
encoded = pd.get_dummies(clean, drop_first=True, dtype=int)
print("Shape after encoding:", encoded.shape)

# 6.4 z-score scaling of numeric attributes
from sklearn.preprocessing import StandardScaler
scaled = encoded.copy()
scaled[encoded.columns] = StandardScaler().fit_transform(encoded)
scaled.head()

> ⚠️ **Leakage reminder:** in a real project you fit the imputer and scaler on the **training set only** (see Lab 03 pipelines).

## 7 · Dimensionality reduction: PCA vs. t-SNE
We project the 64-pixel **digits** data set to 2-D.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

digits = load_digits()
X_pca = PCA(n_components=2).fit_transform(digits.data)
X_tsne = TSNE(n_components=2, random_state=0, init="pca").fit_transform(digits.data)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, X2, name in [(axes[0], X_pca, "PCA (linear)"), (axes[1], X_tsne, "t-SNE (non-linear)")]:
    sc = ax.scatter(X2[:, 0], X2[:, 1], c=digits.target, cmap="tab10", s=8)
    ax.set_title(name)
fig.colorbar(sc, ax=axes, label="digit")
plt.show()

In [ ]:
# How much variance does each principal component explain?
pca = PCA().fit(StandardScaler().fit_transform(digits.data))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker=".")
plt.axhline(0.95, ls="--", color="red")
plt.xlabel("number of components"); plt.ylabel("cumulative explained variance")
plt.title("Components needed for 95% of the variance"); plt.show()

## ✏️ Exercises
1. Which column has the most missing values? Propose a better imputation than the median (hint: group by `pclass` or `sex`).
2. Use the IQR rule on `age`. Are the outliers errors or real passengers?
3. Replace `StandardScaler` with `MinMaxScaler` and `RobustScaler`; compare histograms of `fare`.
4. Load a CSV of your own (`USE_MY_CSV = True`) and repeat sections 2–6.